---
title: "6.1 Fertility and income in 2019 · Complete solution"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/projects/01-fertility-income/02-solution.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/projects/01-fertility-income/02-solution.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If the GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. Each notebook starts afresh.

**Time:** 60–90 minutes, split across sessions. **Before you begin:** Complete Parts 0–5. Attempt the [starter notebook](01-starter.ipynb) before reading the worked analysis.

## You will learn

- Define and audit a complete-case sample from new real public data.
- Produce a labelled chart and a comparable pair of formula regressions.
- Export a table with source, uncertainty and interpretation notes.

## Why it matters

In 2019, did economies with higher GDP per person have lower total fertility rates? This project moves from a documented input to a descriptive figure, two OLS specifications and an exportable table. GDP is a measure of production per person, not individual earnings; total fertility is a period demographic rate, not the completed number of children in a particular family. Start with made-up numbers, then use the real World Bank snapshot. The starter gives explicit steps; try it before reading this complete solution.

## Worked example

### Prepare the analysis libraries

Run this prepared cell before importing analysis packages. It uses the course dependency pins and installs only if required. If it asks for a restart, restart the notebook session and run from the top. Installation needs internet; no paid software is needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. These packages run inside the notebook session; this does not install Python on your laptop.

### Establish the units with a tiny fictional example

Import the existing course tools and inspect these made-up numbers. They demonstrate columns and units, not a real empirical pattern.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
plt.rcParams.update({"font.size": 12})
practice = pd.DataFrame({"economy": ["Northland", "Eastria", "Southport"], "gdp_pc_usd": [1000., 6000., 24000.], "fertility": [5.0, 3.0, 1.8]})
practice["log_gdp_pc"] = np.log(practice["gdp_pc_usd"])
print(practice)

A log predictor describes ratios rather than equal dollar changes. Keep the outcome in births per woman; a log-income coefficient is not a percentage change in fertility.

### Prepare the pinned public-data snapshot

Run this prepared setup. It uses the course files locally or recreates the exact frozen public-data snapshot bundled in this notebook. It verifies each checksum. The source observations are from 2019; the snapshot was retrieved on 2026-11-11. No account, key or private records are needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared project-data setup"
from pathlib import Path
import hashlib
import base64
import zlib

# Frozen public-data snapshots, generated from the rights-documented course inputs.
expected_files = {'fertility_income_2019.csv': '7281d429c61b91021c29554c4a116615a103211b7276eb333274aae1ab3a3048'}
saved_snapshots = {'fertility_income_2019.csv': 'eNqlW1tz4zbWfOevUPnh22wVwiLuwKNuljW6WNHFjv2Soi2OzIwkOpQ0s/av3z6gZ77sZkW64jy4JlUGeAD06dN9AGePxb7Yvfz2WKwzllX/w8pskxd79pKlJdusn397fvztdFizz1l5zLf58SVqd25Zuzw9pGycHvN9q73Lyvwxbf1fq5uW+cNDlu6ZSLhnMjFaxcppLqSW3jIe24RH7csBa3/ePKX7/HDE715M8vV6m7X66eHIWtOiPD612p9pStb60+9h/ln6JfzzoppfeRObROhECe0007GQLmoPrll7vym2KVucHn5epE9picFvE4ZxGGFjZ1xiBZdGCYxUSkTtcYe1tw/4XMr6p7J4zmhJ2f5YpttW+5C/jTaJ8bGSPpFc8kTQqqTXUXvaw3fXRVnWjlZcaHw8UdoZrpXEaPw7as/7bLXPj9m61S7Th1Z/l5fpMTt8YHO0lw7b45S2QoQwhbL4EDa/3CCufN94gN5rHXurLG2WEg5zOIeNmk8wxy5r2ChEYGMhnLMSEykabaL2AmOr7+1bi3RXYAosLQwLi3jMP+eP1QQcY03Mpfaa46gcE7FKVNReYgmIf3NKW+l+3eqk5cNp3bgY7hzwImkiKUQIRxngZbVg7dOBYs9rYtGaexVb4CzBhoStMBrbuVpWoxu2wjuHnfTKSq1FosK3o/Z9n7Vfs/IhzX9HlHXjXaLxcWm5NdpzOkzHo05vyDqn8rRf5+exLlUMmAJrHnAPSWIxsj9mnWy7yU+72s8ay5FiQA/+5Y0KJ9jpTzF2n+/PfpNzmcQOuaUlst5rpmIfdS7bFOwXwK51mR6Ks6OtUbHTMrH4T3llMNrQYgc91kn3m226zg5PbFGcKBN+hCq48LH1OGJADSkKrHCHQXN8dLtJG86HJxJMBV5IFFeJo9O1CPkKo9OnMs0/QFICu47IlHTaexny0CmJuRfsApOnu/TAWsun7KKRTUFVSYyfHswhrAu8Y6PO8Ip1igNyMSTDVVa+ZpviK2V3HYlxIZALWuGQiJwxGYAddcZzwkVang61o520sbU4XaO0kZpCAS90xvc0On/NmtZiuAWUuFLe4qeQdFpg4M5khQnKHWXz2xZXM3zHlcYJgy+BDa2rHXDYymuAudjmX/NGDlAikTFo1wtHExCjGAeYzIHNMn3Nt42MmABmTkphgMnEylDSNCboMKKhdF0cmqYQ3AsqINpwrpynGTjNgKxCKmd5q0fbf0i36e48HUnQsYm1pRJrdMXMHOe3xCxPJ6qq/50gUlsTO+yf5UmiE4pcKSz9Fksvjodv6f58wQTtiDhxXhsQUSJo23xiom77kv0Ax+eK0ufZ8+lhizDPzYUzp6IAQkHpRhllJgbwuu0p6yKE/33wChjDJ5HaAlGEHEIp61712eJbfgSDbgH9OsA67bACbxMhE6l04F+HCYasi8zdZ9vW8EBz1KM+wfrBhh7lTOL4CX4ak4wxSb5txDyh3cQen0ceSyMoBonhUxq+ryuDCJk2PMHylQzZhpmi7vCGdYtj1lr/Y/i1yMvsfA3gCiBD0iLtEiQRGBVJH3Unc+w5wi2KGi7XkAACcE8UiNlaYmPNo+51j110C+gs1uplu5jOPb44OwskWkzc6pHlwhmcudAI4HrwY5L68SjXLlZCGWOF8pxqAqReF4nfLbbF7qE58xG7iDWOHrmvkLIh7TDDBDPsihJpe34HOBBrpDNKorA4fJvTx2fY/vShaN1k5fr83oP3k9g6zsGWnFe6wWP0HNArUCZac/rVJugIKCAURecU1IvioQ53Vx3WfYcK90IKxI9z91JD+xDVK4y+xegyfUyLxq97IRLwpEH+ORmYA1+/w86lLzss93vqNEwD5Ug1FrrZWJsoFUoOduJuxrovzw0VRyAGG3NSIc6GegP4QD51X7PHp/raLlRiBA4bCcSlDIobJ9/rr9gApSbdv9QqIOgtSFDvoWGhg6u9630ast7v+QMYNv+AMKAiyg0qilQJlAFIFewc9SZt1it2+f4duPDIBRROqGtvhLTVliK+6Yj1sv0uLb/UrU17qGkgw0hkhBVhY6IeMuL75//E5k1nC1zFCUcQJkkgK7AWaUzUu2/DUW0yUl9/38hApcdKSwUBL3EaVHyge/rdFes/nlBxy8bcF1DPCeof+BNlh2aA3ov6gzt20d+8PCOeYLoqDvr7B4pDiCH0vaAyk5CocYgTmd4v82OZna+vTFZLWszY4pnUZm0mwI2Bv1FGHCAdfJ3E0CXrH47FvjEXBCedi4ogoR0olU3UX16x/vEpL57zGg3gJYouQQUU5H2gYBThy+GUXeb7pgKsnNQOipxUvUQRqmrnJRLpMv89r7FdYLwYOkvBxVr4SQKWw0AItktE+JjVfhRK0cXQGk4jdzV91BmMvWaXaVlk76r58CHgbxStxIG4OFlQjlO9hI0FUB7LYp/RL7cus3XcWhwPANB50aYdUkyYypJbklGAdDRod9gAleR8EbYKBZzj3BVFwgEXVIJo0Jl/7xmM8v1mXdR7OckNsTgkF+ScJ92JxI8G/WvwYFFuGgysRQRgLJhHSKmQQYJHg6s2GzzVKUdgBQZSOygIhxokETrKUDQYYsn5A75xTMu67yJKzxElUDY45fuaHIJKAqZg30ijAqckcqyJBpMOuxikQSJUNuvsJksCqAFOoJGBVwI4Ip123r78cyeHKj+dHY8aHwtoa4Au+EjSCZom+IX1/zilxwIsuG01rIL6LMToIBDE40hsKOGjwbzLBmWW1eOde6hDEG8iINKhlinJAJN5j8YGbd1U6RNFpRL+BGIB5j2UO+z+fBy+3pjlGJJAK4FcPfbAUBmA7IgGywn2MD1mO5iaRoumsY/eO+m1FiZkfEDMKsxRZ4lgDFTsgXMIRTAU1QnotMHqDgNf0v07JCLKhBNKgW3ArlSQNUB+NYJKvYJKbY3ox6I9Z62g2GsynejOQ/Fy7nWiQ+czMSq6mvYYZlpDdzXbRJ2A67AkJQUBmqwqRzRzyM6ywNj6hoaGL4+thNX1xghdOaar5ZBdpTlUSxMQyG/B9AXJroM2cUZEV6spuzrtN2lZK5rIZWCA9jDZkgeq0S4a9qZsuF+/Eeb5reMJXBIo24CvwJj4NMp3NJxg9AFlufjcmtQ3y+BwoRW1BstZq0KzDbo9GmLz8X0qcf/dO8IykWbOUG+FOslYu4iGQP2wzBqtJXf4mjGu6nOGkuoxeMouhshuFqrMLn/8qLrA3Ba+T5KDoRjJ0WFR818QZPrH359XG0vWAvJeCq8ECBrOeLjA2h8b127AFCZGwsEZ4bSowFqFwXMcVZlm24+IvsRTtbYuEbANipG8jIbLNhse020t+qR0QJ/FcYAGDBkuYaNP7Qn7lO7Sd4hqoI7qpPNGgQVFkCo++nQ9Z5+Kcv2h+wqtLdgVJsSCYQWJCWRI9Gk2RWzPBOrzSQFOw3Ygq5O3WwNpolH7no3S1/TLU7hHqdkTD8UBTBtUR225pPrtfDTqT9ko27/UFFUPzQARlTinwalUvhWK4miAD7+Um5fX/3cIdXwA1wpKBz/DnfCEMIY5rshA7h6KdR0bCJw+VDQMuwoYJZMkdDQaztkox7HhKGsaJ/heLEgFSinhj/Bd1NLRtM0Wxxia6Xg8hJ7pNPuaN/MxXUPEVJmFhB8nWCCgaARcXIwKUvdvGX6+QElIIFAbdhCHiKKQQNtghtslG52+gZY/QA6gXIt9klQuwHykz5yIxu1rLKtozXrzmi3WzqPQJI6a4LRLKHsAx7gzZePsId0XH0A8HAqOzyU4PtjnMDdKGuaes3H+EHzhWSVEfgEpSJYBYoIaT0LT0Dsa+vIBQwmSlrD0wHPiFAk9LBgZPu5WuBifHpvbSZzTNYXEYSawRFU3VahoPOwjuuzx6ZjtD8es3slxA/OPU5NeYoHcVE3J8QhhlHlrnO6//LVUIReRSmTXYQe4VEGI46AXOOjsUByfihqB7AS54MTAg2kefDSFvFwh5OPTqenek7jAgb5gIEBk4cpAyWi8+pWNT//KkMinclM7HvxDF7YKmPAmNCugTsc3bdrrrw3fBmFC21HYSVIpW8OjSbsLFFAL632qDKUaW2Do0kJIh1kSsKDHNJfh4CdpSWf+0yXE8uNT6xn/+8/m3hi4hbQdMKVQM0IThyKbs0lRFo+PxQeSWmO/E08dKBE6KpDik+41Jt6nmLfeCFhlwJSeMB4EDTAaTXptDN6ui6/1Xo90sKR+CzcCQpKqusPgAZvAQ2zSwyM829l+p/cx7bO0BtQNgCJtIaInvRuM3q7zr9nhL6BGMikQtaXuhLdB20AsT/q/skn2r/yxuUEJZ+5jwiS1OIUNSRFNrsb4ZHl4Srd/au+fbTQIjx2WJFmpe2OC4MAko97bhQRwlq2b+ivQUo76tNRU00q7CuOT8ZAWf/6SFo4cSe0l7Kd0iacHCc7j4+MljTt+gOtATlR7EzhayzE9o35hNJkAnrBEOzLfZ6unQukFgOAAOcE8CHETTaZ9QiAILtuURb0Op6segNdbSX03ar9g+ICGb4ra23ZYMCh4ZcjUK5Q1BvBOZ9VJZOWeUjWHo2s+VhEuXYyBkRO0FcT1Gil/fY8oXqkt8MfpfPOeOv84DkDCc7qx0dTjjCZzOpZTmR8DZZ7lW2tsnFhhYWgEdQO8w86vFm9j81PNlQOHtKArEgDSw5ES1SKHbgOM0m/ngaQdKM0hWLCGUFZTY1EDSHeLMPKl1nlxLDVBYUTgGE/3UygU0RTKeQoD81CzVGUlQCbotJD51J+idxrT7phNs2/I1u33zDkvjRSnnr7GcBgS8pvwfNP+nE3zTXaebTR1paAUqGWTCODEoJQj5kG7GlgTM0IlpoLrA0J4aPc41JTpsIuhYLh0c2pWAXBiMXWXuYZH94GkQXbTMUgjI6Q29hdBWVrR3R5yhatwswGRM4WoBNa/pbVGB05CQLkl1LFKQFY0VkbTGe36c7r9K82Gy2+FSgoNoJJKJEbT+QrnW2SUy+eR4RJCQ6INSrckWMHYR9P76oDvs7RyimfTWdD1qzDYbmR2uL+1IrqGpb/efcRQAasAPGhNhQOgFaG6R7P2iH3/zQ9MLj28D13QeIcCRYt2HHNPMfceVrIRHHRdSvfkRpB1oaoknItmAPUsK09Nw20iQR+h6+yFSYLSc1gbqtrsKd/mz8/5PqthPqkSunj09MQC5oDGo5hHs/Et4t+mp5rT1g5phQIEElbVuy2irhmYe5Y+n9IWnflbH7PGVVgNxAhLD2pkcI4SQnV2jfCLpr4C9ZjpjhqZBa9VXdcjgPmQzU5ZeSzo0rRo/bRaNOozZDncPMABXUL0Qp4Lxmg2H/1wbeH6epYVz9vsH4cmD0c1DP5zhhIwA5ROG+RZ3X0LXTjCqgvnbZLIHwu5w04GhnlpWgC0BEo2CI5e0JnKnVEAiz67zRBkBzYhmNhB+voRnUDXnCRcLKfrGR769ED73SV7k8M4tZeG1h3w5YJ5FIkyLqksso1+aS/ZLyk1+v92eMYQJGGbIbJBlOHRDAhofr1i82LX6FwEchiGIQHpQRCpUNmiOUrx/HTAb+3p9iYrcRBFvWHjCrKEtgcyr+JqlLn5bZvNv+EIztcaJwACyFMEYOgnlWWno0V7xRbpaZ2H68eP3JEKr40JrSWcoAr9HeWjRW+KkKhldfbOA/sCV0UNZnqS5el1B7fRoo+BJPCoipzTKEoGYALaqHmahvJoMZixRb7fpM/IrfNAMfSOIqaMgDaS4SEXVOli3EHN2ha7Yv8OYSewp6j8ONnwLoTYGTP08fmsLNPWOCv254WdQ1ZaMAu3OA1OvQUOvb0Y37D+trVIt1/fc62sJMklDs0Pn2Oql30SQUBcL9JKo+6L+ueVhhpCkHjakuKjWwKUr8X1hF0sAOvtj4vFhtc1EqcIeiC7hnqlSAVhmjm2Mysf6nODXtLRRaCn5y3hfQP2YdF7Ew/16CFvh+8sceaw4MtilwUmmmHdj/lzVtNVlDoWZHc46QnKhmixwq5BF6OyNr8ehKWApkfo1OENzoSuohc3I7bYwtx+eV9X0iMbITOdN3Q5LSrptbiZhkmaHjcLiE6gBnUdANBBLUM9Lm7pSRrIpJZHNKSBI2slHGlHVb1IWdzes/7hGy39vMCX0PbIFboMh+iWVRMHTPLrhPLuCNClJaxZ66fe6fjeHgY41dB7ZDgF2ovQhkI4d13A5+XxKdtusxqr4ulxmYa3JN7hQQLTYJzmC3m0Hy8rqvP4+2oM7oTadMRwdMcbHle4aNlts+Wp/FI1crspdMHhvQ+SJLeOtAGHHHbeVU0liyl79CpwfZ48vKRnuAraUEkpNNJNcREtB9dsWWzON+BgrHDOqIvUgDOBbJWmJwrR8gqreErzehVtTaJJ3+LwPT2GoJsNVJHlpxFbpr/nb3q3/mKMuoXSIvdQRynxBE56OZqEPaQH/U1T4Iw13ekKFGGtwr2gVNFyvGDLfFeUP4+hSbI6aWnoZZCCz3PIflKFYKxoeT3F1qFs1FgI6xw9vNUkLSBQKXaLLy+x6WCNfJ2uAwSWxUO6ae4YWcHpPlXQax1uwkU3oWmFOE7YhY/UYok0AjZFeGBjPV1gYo9Bb7TH+UvtDb6Xnv6KBSWRHquFXHQeg28w+Gu6rRHtGkQGAScMaiv9jQDpbSTi8h7ISvev9Z0KFGFS57pqNlZW2EQruOjVpkHY0DUx8IBSDl9IhdRC4K5Gc7b6Qu/ma1cLvvBkuuk1TPguPaBw0QryeFWe3qOOoe65pmtumCX7Vr+gOFeL9veXMYtj+DOa//Go2Gh6UujJ0oJcZHjzn2DV9x22en3ImtMJEIL7h5SBZgYyAyUZH910l6GdfIMqiDEBlcenrFW9wQimrek1HbVDNXSNFpAYpnq3pKIbqLKLG8iy11O2peumzkXzIwIhQ7qiXHgdXhxxoOJm0GEd6kEdnhBmucnf/YiT/JPCBMM5+8+BcGPxIm6uNfAwsBbUmIZqCvUr0dHNdILpsmNrWvvAQ2FbfJIITX+4EMyUVzK6WS3ZTbo/pcea9CB0xgaOVqFqcEdm3vrodoGyWf83SEpq+gxG4lvABxCOE4l+Hf3KRsWh+Fov8BQnxqaHKRCIFcvo6K4PfXeXgW7ZRx8FkF5PZHRPNxhVt6cadf5STZLw5dSqo64ENQWArPtJh91Xb6TOcgQET0x/eeKdosVQ8UKu3kPw3Oe7h/ThW937c0dunsMBgfcTMpdWuejftYa3qQ=='}
data_paths = {}
for name, expected in expected_files.items():
    destination = Path('data') / name
    if destination.exists():
        raw = destination.read_bytes()
    else:
        raw = zlib.decompress(base64.b64decode(saved_snapshots[name]))
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError(f'Changed project input: {name}. Review its source before continuing.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(raw)
    data_paths[name] = destination
print('Verified project data:', ', '.join(data_paths))

Look for **Verified project data**. The [data dictionary](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md) records raw API URLs, licenses, transformations and limitations. A frozen snapshot makes the exercise reproducible; it does not make the observations current.

### Inspect the complete input before selecting a sample

Load the snapshot, inspect the columns and count unavailable indicators. Each row is an economy-year. World Bank geography includes countries and territories, and source region/income aggregates have already been excluded using documented metadata.

In [ ]:
data = pd.read_csv(data_paths['fertility_income_2019.csv'])
print(data.head())
print("Input rows:", len(data))
print(data.isna().sum())
print("Duplicate economy-years:", data.duplicated(["economy_code", "year"]).sum())

All 217 economy records remain in the input. There are 6 unavailable GDP values. The other indicator is complete. Missingness is not a zero value and may be systematically related to the question. Both indicators are for 2019; the raw source snapshot and region labels were retrieved in 2026.

### Define the analysis sample

Select complete pairs without changing the original input. Check year, unique economy keys and valid measurement ranges. Check positive GDP before taking its natural logarithm. Explain the observations excluded, and retain the complete input for an audit.

In [ ]:
sample = data.dropna(subset=['gdp_pc_usd', 'fertility']).copy()
assert sample["year"].eq(2019).all()
assert not sample["economy_code"].duplicated().any()
assert sample["gdp_pc_usd"].gt(0).all()
sample["log_gdp_pc"] = np.log(sample["gdp_pc_usd"])
assert sample["fertility"].ge(0).all()
print("Complete analysis rows:", len(sample))
print("Regions:", sample["region"].nunique())

The analysis has 211 economies across seven source regions. All receive equal weight, so it describes economy-level associations rather than a world-population average. Complete cases need not represent economies with unavailable data. Checks can reject impossible inputs, but plausible ranges do not prove accurate measurement.

### Make the figure communicate the question

Use one point per analysis economy, label both units and 2019, and retain the selected sample. A logarithmic income axis lets equal income ratios have equal spacing. Save the exact named Figure, not an unrelated current plot.

In [ ]:
#| label: fig-project-fertility
#| fig-alt: "2019 scatter of 211 complete-case economies: higher log-scaled GDP per person is broadly associated with lower total fertility, with substantial differences among economies."
#| fig-cap: "World Development Indicators, 2019 observations; snapshot retrieved 2026-11-11. CC BY 4.0; World Bank, UN Population Division and national/OECD/Eurostat providers. Equal economy weights; descriptive association."
output_dir = Path("downloads/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
figure, axis = plt.subplots(figsize=(6, 4), layout="constrained", facecolor="white")
axis.scatter(sample["gdp_pc_usd"], sample["fertility"], color="#21645a", alpha=0.8, edgecolor="white", linewidth=0.4)
axis.set_xscale("log")
axis.set(xlabel='GDP per person (current 2019 US dollars; log axis)', ylabel='Total fertility rate (births per woman)', title='Fertility and income · 2019')
axis.grid(axis="y", alpha=0.25)
figure.savefig(output_dir / "project-fertility.png", dpi=300, facecolor="white")
plt.show()

The figure shows an association in the retained sample. It does not identify the causal effect of income on fertility or an individual family’s behaviour. GDP is current US dollars for 2019, not the Gapminder extract’s historical inflation-adjusted measure.

### Fit two comparable descriptive specifications

Use the same complete sample for a simple model and one adding region categories. Keep HC1 t-based approximate intervals explicit. These broad region labels come from the 2026 metadata snapshot. They are controls, not a causal identification strategy.

In [ ]:
simple = smf.ols('fertility ~ log_gdp_pc', sample, missing="raise").fit(cov_type="HC1", use_t=True)
adjusted = smf.ols('fertility ~ log_gdp_pc + C(region)', sample, missing="raise").fit(cov_type="HC1", use_t=True)
print("Simple coefficient:", simple.params['log_gdp_pc'])
print("Region-adjusted coefficient:", adjusted.params['log_gdp_pc'])
print("N:", int(simple.nobs), int(adjusted.nobs))

The simple slope is about -0.657 and the region-adjusted slope about -0.414. The unit is births per woman per natural-log unit of GDP per person. Multiply by log(2) for a fitted comparison associated with twice the income. HC1 allows unequal error variances; it does not account for all spatial dependence or source-estimate uncertainty. Seven regions would be too few clusters to casually treat region clustering as a universal fix.

### Export a table with its interpretation notes

Extract results from fitted objects instead of retyping estimates. Include N, year, term and units. The selected-coefficient table intentionally omits intercept/category estimates, so the notes must state that scope.

In [ ]:
rows = []
for label, result in [("A: simple", simple), ("B: region controls", adjusted)]:
    interval = result.conf_int().loc['log_gdp_pc']
    rows.append({"model": label, "term": 'log_gdp_pc', "estimate": result.params['log_gdp_pc'], "HC1_standard_error": result.bse['log_gdp_pc'], "ci95_lower": interval.iloc[0], "ci95_upper": interval.iloc[1], "N": int(result.nobs), "year": 2019, "unit": 'births per woman per natural-log income unit'})
results_table = pd.DataFrame(rows)
results_table.to_csv(output_dir / "project-fertility.csv", index=False, float_format="%.8f")
(output_dir / "project-fertility-notes.txt").write_text('2019 observations; snapshot retrieved 2026-11-11. 211 complete-case economies from 217 retained input records; equal economy weights. Selected coefficient only; outcome/predictor unit: births per woman per natural-log income unit. HC1 standard errors and t-based approximate 95% intervals. Model B adds seven-region categories; category coefficients and intercepts are not displayed here. Region metadata are from the retrieval date. Descriptive association, not a causal effect or an individual-level relationship. Source: World Bank WDI; UN Population Division, national statistical offices and Eurostat (fertility); national accounts, OECD and World Bank (GDP). CC BY 4.0 with World Bank additional terms; no endorsement. Source estimates and definitions differ; missingness, measurement uncertainty and spatial dependence limit interpretation. See the course data dictionary for exact API requests, raw hashes and transformations.', encoding="utf-8")
print(results_table[["model", "estimate", "HC1_standard_error", "N"]].round(4))

Download the [figure](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/project-fertility.png), [selected-coefficient CSV](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/project-fertility.csv) and [notes](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/project-fertility-notes.txt). In Colab use the Files panel. The source data have CC BY 4.0 terms and provider attribution; the course’s original presentation has no selected license. A complete research handoff should also retain the input snapshot, code, dependency file and rerun instructions.

## Try it

Work in your own copy. Add a cell for each task and attempt it before reading a solution.

### 1. Report retained observations by region, with the correct denominator.

<details><summary>Solution 1</summary>

Run this solution after your own attempt.

In [ ]:
region_summary = sample.groupby("region").agg(n_economies=("economy_code", "size"), mean_outcome=('fertility', "mean"))
print(region_summary.round(2))

The group counts sum to 211, the analysis sample, rather than 217 input records. These are equal-economy means of the retained sample, not population-weighted regional estimates.

</details>

### 2. Translate twice the income into the outcome’s units.

<details><summary>Solution 2</summary>

Run this solution after your own attempt.

In [ ]:
translated_comparison = simple.params['log_gdp_pc'] * np.log(2)
print(translated_comparison)

The simple-model comparison is about -0.456 births per woman for twice the GDP per person. This is not an individual income intervention or a percentage change in fertility.

</details>

### 3. Read back the export and verify its sample and selected coefficients.

<details><summary>Solution 3</summary>

Run this solution after your own attempt.

In [ ]:
reloaded = pd.read_csv(output_dir / "project-fertility.csv")
print(reloaded[["model", "N", "year"]])
print(np.allclose(reloaded["estimate"], results_table["estimate"], atol=1e-8, rtol=0))

The values agree within the chosen eight-decimal export rounding. Explain why adding region changed the specification while leaving the selected sample unchanged. A smaller coefficient is not evidence that the adjusted model has removed every confounder.

</details>

## When it goes wrong

These demonstrations catch a specific intended error so the notebook continues. Unexpected errors elsewhere should still stop execution.

### A familiar column name belongs to another dataset

This input uses its own documented field names.

In [ ]:
try:
    data['gdp_pc']
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect the actual columns and data dictionary. Renaming a variable does not establish that its units or population match another source.

### A non-unique lookup would duplicate observations

Create a deliberately duplicated key and reject a many-to-one merge that does not satisfy its contract. This changes only an in-memory practice copy.

In [ ]:
try:
    bad_lookup = pd.concat([data[["economy_code", "region"]], data[["economy_code", "region"]].iloc[[0]]])
    data[["economy_code", "year"]].merge(bad_lookup, on="economy_code", validate="many_to_one")
except pd.errors.MergeError as error:
    print(f"{type(error).__name__}: {error}")

Resolve the meaning of duplicate source keys before joining. Dropping arbitrary duplicates simply to silence this error could hide a substantive mismatch.

## Recap

- Start from documented raw provenance and retain the unfiltered input.
- Use the same explicit sample for model comparisons.
- Keep units, uncertainty assumptions and omitted coefficients visible.
- Association across economies does not establish individual or causal effects.

**Next:** [6.2 Internet use and urbanization](../02-internet-urban/01-starter.ipynb).